In [1]:
"""
============================================================
  Hate Speech Detection — SOTA Pipeline
  Model  : LLaMA 3.1 8B  +  LoRA (QLoRA 4-bit)
  Data   : HateXplain  +  Davidson (hate_speech_offensive)
  Author : Final Year Project
============================================================

REQUIRED INSTALLS (run once in your environment):
  pip install transformers datasets peft bitsandbytes \
              accelerate scikit-learn torch tqdm

DATASET LINKS:
  HateXplain : https://huggingface.co/datasets/Hate-speech-CNERG/hatexplain
  Davidson   : https://huggingface.co/datasets/tdavidson/hate_speech_offensive
  GitHub orig: https://github.com/t-davidson/hate-speech-and-offensive-language

HARDWARE:
  Minimum: 1x GPU with 16GB VRAM (e.g. RTX 3090, A100, T4 on Colab)
  Recommended: Google Colab Pro+ (A100) or Kaggle (T4 x2)
  With QLoRA 4-bit: can run on 12GB GPU

HOW TO RUN:
  python hate_speech_detection_llama.py
  OR on Google Colab: paste each section into a cell
============================================================
"""

# ─────────────────────────────────────────────
# 0. IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────
import os
import re
import numpy as np
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.metrics import (
    classification_report,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ── New imports needed (add at top) ──────────────
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score
import numpy as np

# ── Config ─────────────────────────────────
MODEL_ID        = "GroNLP/hateBERT"
# If you don't have Llama access, use this open alternative:
# MODEL_ID      = "mistralai/Mistral-7B-Instruct-v0.3"

OUTPUT_DIR      = "./llama_hate_speech"
MAX_LENGTH      = 128       # max token length per sample
BATCH_SIZE      = 32         # reduce to 2 if OOM
GRAD_ACCUM      = 1         # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
EPOCHS          = 4
LR              = 2e-5
SEED            = 42

# Label mapping (unified across both datasets)
# 0 = hate speech  |  1 = offensive  |  2 = neither/normal
LABEL_NAMES     = ["hate speech", "offensive", "neither"]
NUM_LABELS      = 3

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: Training on CPU will be very slow. Use a GPU environment.")


# ─────────────────────────────────────────────
# 1. LOAD & PREPROCESS DATASETS
# ─────────────────────────────────────────────

def load_davidson():
    """
    Davidson et al. 2017 dataset.
    Labels: 0=hate speech, 1=offensive language, 2=neither
    Already aligned with our target label scheme.
    """
    print("\n[1/4] Loading Davidson dataset...")
    ds = load_dataset("tdavidson/hate_speech_offensive", split="train")

    texts, labels = [], []
    for row in ds:
        text  = row["tweet"]
        label = row["class"]   # 0=hate, 1=offensive, 2=neither
        # Basic cleaning
        text = re.sub(r"http\S+", "[URL]", text)
        text = re.sub(r"@\w+", "[USER]", text)
        text = text.strip()
        texts.append(text)
        labels.append(label)

    print(f"  Davidson loaded: {len(texts)} samples")
    _print_distribution(labels, "Davidson")
    return texts, labels


import json, urllib.request

def load_hatexplain():
    """
    Load HateXplain directly from GitHub raw JSON files.
    Bypasses the broken HuggingFace loading script entirely.
    """
    print("\n[1/4] Loading HateXplain dataset from GitHub...")

    DATASET_URL   = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json"
    DIVISIONS_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/post_id_divisions.json"

    # Download the JSON files
    with urllib.request.urlopen(DATASET_URL) as r:
        data = json.loads(r.read().decode())

    with urllib.request.urlopen(DIVISIONS_URL) as r:
        divisions = json.loads(r.read().decode())

    label_map = {"hatespeech": 0, "offensive": 1, "normal": 2}
    texts, labels = [], []

    # Iterate over all post IDs across all splits
    all_ids = (
        divisions.get("train", []) +
        divisions.get("val", []) +
        divisions.get("test", [])
    )

    for post_id in all_ids:
        if post_id not in data:
            continue
        row = data[post_id]

        # Reconstruct text from tokens
        tokens = row["post_tokens"]
        text   = " ".join(tokens)
        text   = re.sub(r"http\S+", "[URL]", text)
        text   = re.sub(r"@\w+", "[USER]", text)
        text   = text.strip()

        # Majority vote across 3 annotators
        annotator_labels = [a["label"] for a in row["annotators"]]
        mapped = [label_map.get(l, 2) for l in annotator_labels]
        label  = max(set(mapped), key=mapped.count)

        texts.append(text)
        labels.append(label)

    print(f"  HateXplain loaded: {len(texts)} samples")
    _print_distribution(labels, "HateXplain")
    return texts, labels


def _print_distribution(labels, name):
    from collections import Counter
    c = Counter(labels)
    total = len(labels)
    for k, v in sorted(c.items()):
        pct = 100 * v / total
        print(f"    Class {k} ({LABEL_NAMES[k]}): {v} samples ({pct:.1f}%)")


def combine_and_split(t1, l1, t2, l2, test_size=0.15, val_size=0.10):
    """Combine both datasets, deduplicate, and split into train/val/test."""
    print("\n[2/4] Combining datasets and splitting...")

    all_texts  = t1 + t2
    all_labels = l1 + l2

    # Deduplicate by text
    seen = set()
    unique_texts, unique_labels = [], []
    for t, l in zip(all_texts, all_labels):
        if t not in seen:
            seen.add(t)
            unique_texts.append(t)
            unique_labels.append(l)

    print(f"  Total after dedup: {len(unique_texts)} samples")

    # Split: train / val / test
    X_train, X_test, y_train, y_test = train_test_split(
        unique_texts, unique_labels,
        test_size=test_size, random_state=SEED, stratify=unique_labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=val_size / (1 - test_size), random_state=SEED, stratify=y_train
    )

    print(f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    return X_train, y_train, X_val, y_val, X_test, y_test


# ─────────────────────────────────────────────
# 2. TOKENIZER & PROMPT FORMATTING
# ─────────────────────────────────────────────

def build_prompt(text: str, label: int = None) -> str:
    """
    Instruction-tuned prompt format for LLaMA / Mistral.
    During training we append the answer; during inference we leave it open.
    """
    instruction = (
        "You are a content moderation assistant. "
        "Classify the following social media post into one of three categories:\n"
        "0 = hate speech (targets a group with hatred)\n"
        "1 = offensive language (rude but not targeted hate)\n"
        "2 = neither (normal content)\n\n"
        f"Post: {text}\n\n"
        "Answer with only the number (0, 1, or 2):"
    )
    if label is not None:
        return instruction + f" {label}"
    return instruction


# ── Replace tokenize_dataset() ───────────────────
def tokenize_dataset(tokenizer, texts, labels, max_length=MAX_LENGTH):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt",
    )
    return Dataset.from_dict({
        "input_ids":      encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels":         labels,
    })




# ─────────────────────────────────────────────
# 3. MODEL SETUP — QLoRA 4-bit
# ─────────────────────────────────────────────

# ── Replace load_model_and_tokenizer() ───────────
def load_model_and_tokenizer(model_id: str):
    print(f"\n[3/4] Loading model: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
    )
    model.to(DEVICE)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M")

    return model, tokenizer

# ─────────────────────────────────────────────
# 4. METRICS
# ─────────────────────────────────────────────

# ── Replace compute_metrics() ────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    macro_f1       = f1_score(labels, predictions, average="macro", zero_division=0)
    return {"macro_f1": macro_f1}


# ─────────────────────────────────────────────
# 5. TRAINING
# ─────────────────────────────────────────────


# ── Replace train() ──────────────────────────────
def train(model, tokenizer, train_dataset, val_dataset):
    print(f"\n[4/4] Starting training...")

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=50,
        report_to="none",
        seed=SEED,
        optim="adamw_torch",
    )
    from transformers import EarlyStoppingCallback

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    model.save_pretrained(f"{OUTPUT_DIR}/final_model")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
    print(f"\nModel saved to {OUTPUT_DIR}/final_model")

    return trainer



# ─────────────────────────────────────────────
# 6. EVALUATION
# ─────────────────────────────────────────────

# ── Replace evaluate_on_test() ───────────────────
def evaluate_on_test(model, tokenizer, X_test, y_test):
    print("\n===== TEST SET EVALUATION =====")
    model.eval()

    test_ds   = tokenize_dataset(tokenizer, X_test, y_test)
    collator  = DataCollatorWithPadding(tokenizer)

    # batch inference
    from torch.utils.data import DataLoader
    loader = DataLoader(test_ds, batch_size=64, collate_fn=collator)

    all_preds = []
    for batch in tqdm(loader, desc="Evaluating"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)

    from sklearn.metrics import classification_report, confusion_matrix
    print("\nClassification Report:")
    print(classification_report(y_test, all_preds, target_names=LABEL_NAMES, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, all_preds))
    macro_f1 = f1_score(y_test, all_preds, average="macro")
    print(f"\nMacro-F1: {macro_f1:.4f}")
    return all_preds


# ─────────────────────────────────────────────
# 7. INFERENCE — predict new text
# ─────────────────────────────────────────────

# ── Replace predict() ────────────────────────────
def predict(model, tokenizer, texts: list):
    model.eval()
    enc = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**enc).logits

    label_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    return [
        {"text": t, "label_id": int(l), "label_name": LABEL_NAMES[int(l)]}
        for t, l in zip(texts, label_ids)
    ]


# ─────────────────────────────────────────────
# 8. LOAD SAVED MODEL (for inference only)
# ─────────────────────────────────────────────

def load_finetuned_model(adapter_path: str):
    """
    Load a previously saved fine-tuned model for inference.
    Use this after training is done — no need to retrain.
    """
    from peft import PeftModel

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    print(f"Loaded fine-tuned model from: {adapter_path}")
    return model, tokenizer


# ─────────────────────────────────────────────
# 9. MAIN PIPELINE
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # ── Step 1: Load datasets ──────────────────
    t_dav, l_dav = load_davidson()
    t_hxp, l_hxp = load_hatexplain()

    # ── Step 2: Combine & split ────────────────
    X_train, y_train, X_val, y_val, X_test, y_test = combine_and_split(
        t_dav, l_dav, t_hxp, l_hxp
    )

    # ── Step 3: Load model & tokenizer ────────
    model, tokenizer = load_model_and_tokenizer(MODEL_ID)

    # ── Step 4: Tokenize ───────────────────────
    print("\nTokenizing datasets...")
    train_ds = tokenize_dataset(tokenizer, X_train, y_train)
    val_ds   = tokenize_dataset(tokenizer, X_val,   y_val)

    # ── Step 5: Train ──────────────────────────
    trainer = train(model, tokenizer, train_ds, val_ds)

   # ── Step 6: Full evaluation on all splits ──
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

    def get_preds(model, tokenizer, texts, labels):
        ds      = tokenize_dataset(tokenizer, texts, labels)
        collator = DataCollatorWithPadding(tokenizer)
        from torch.utils.data import DataLoader
        loader  = DataLoader(ds, batch_size=64, collate_fn=collator)
        all_preds = []
        model.eval()
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.no_grad():
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        return all_preds

    def print_full_metrics(split_name, y_true, y_pred):
        acc = accuracy_score(y_true, y_pred)
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        print(f"\n🔹 {split_name}")
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  Precision: {p:.4f}")
        print(f"  Recall:    {r:.4f}")
        print(f"  F1-score:  {f1:.4f}")
        print(f"\n  Per-class breakdown:")
        print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

    print("\n" + "="*50)
    print("📊 FINAL EVALUATION — ALL SPLITS")
    print("="*50)

    train_preds = get_preds(model, tokenizer, X_train, y_train)
    print_full_metrics("TRAIN SET", y_train, train_preds)

    val_preds = get_preds(model, tokenizer, X_val, y_val)
    print_full_metrics("VALIDATION SET", y_val, val_preds)

    test_preds = get_preds(model, tokenizer, X_test, y_test)
    print_full_metrics("TEST SET", y_test, test_preds)

    # ── Step 7: Quick demo inference ───────────
    print("\n===== DEMO PREDICTIONS =====")
    demo_texts = [
        "I hate all people from that country, they should leave.",
        "This movie was absolutely terrible, complete garbage.",
        "What a beautiful day to go for a walk in the park.",
    ]
    results = predict(model, tokenizer, demo_texts)
    for r in results:
        print(f"[{r['label_name'].upper()}] {r['text'][:80]}")

    print("\nDone! Fine-tuned model saved to:", OUTPUT_DIR)

# ─────────────────────────────────────────────
# GOOGLE COLAB QUICK START
# ─────────────────────────────────────────────
"""
If running on Google Colab, paste this at the top of your notebook:

!pip install -q transformers datasets peft bitsandbytes accelerate scikit-learn

from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # set in Colab secrets

# Then run the script:
# exec(open("hate_speech_detection_llama.py").read())

NOTE: You need a HuggingFace account and access to LLaMA 3.1:
  1. Go to https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
  2. Click "Request access" and wait for approval (usually < 1 hour)
  3. Generate a token at https://huggingface.co/settings/tokens
  4. Store it as HF_TOKEN in Colab Secrets

ALTERNATIVE (no access needed):
  Change MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
  Mistral is open-access and performs nearly identically.
"""

Using device: cuda

[1/4] Loading Davidson dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.63M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24783 [00:00<?, ? examples/s]

  Davidson loaded: 24783 samples
    Class 0 (hate speech): 1430 samples (5.8%)
    Class 1 (offensive): 19190 samples (77.4%)
    Class 2 (neither): 4163 samples (16.8%)

[1/4] Loading HateXplain dataset from GitHub...
  HateXplain loaded: 19229 samples
    Class 0 (hate speech): 5935 samples (30.9%)
    Class 1 (offensive): 5480 samples (28.5%)
    Class 2 (neither): 7814 samples (40.6%)

[2/4] Combining datasets and splitting...
  Total after dedup: 43772 samples
  Train: 32828 | Val: 4378 | Test: 6566

[3/4] Loading model: GroNLP/hateBERT


...

Evaluating: 100%|██████████| 69/69 [00:32<00:00,  2.15it/s]



🔹 VALIDATION SET
  Accuracy:  0.8209
  Precision: 0.7969
  Recall:    0.7917
  F1-score:  0.7938

  Per-class breakdown:
              precision    recall  f1-score   support

 hate speech     0.7186    0.7401    0.7292       735
   offensive     0.8522    0.8744    0.8632      2453
     neither     0.8197    0.7605    0.7890      1190

    accuracy                         0.8209      4378
   macro avg     0.7969    0.7917    0.7938      4378
weighted avg     0.8210    0.8209    0.8205      4378



Evaluating: 100%|██████████| 103/103 [00:48<00:00,  2.13it/s]


🔹 TEST SET
  Accuracy:  0.8136
  Precision: 0.7913
  Recall:    0.7819
  F1-score:  0.7860

  Per-class breakdown:
              precision    recall  f1-score   support

 hate speech     0.7215    0.7287    0.7251      1102
   offensive     0.8416    0.8725    0.8568      3679
     neither     0.8109    0.7445    0.7763      1785

    accuracy                         0.8136      6566
   macro avg     0.7913    0.7819    0.7860      6566
weighted avg     0.8131    0.8136    0.8128      6566


===== DEMO PREDICTIONS =====
[HATE SPEECH] I hate all people from that country, they should leave.
[NEITHER] This movie was absolutely terrible, complete garbage.
[NEITHER] What a beautiful day to go for a walk in the park.

Done! Fine-tuned model saved to: ./llama_hate_speech


'\nIf running on Google Colab, paste this at the top of your notebook:\n\n!pip install -q transformers datasets peft bitsandbytes accelerate scikit-learn\n\nfrom google.colab import userdata\nimport os\nos.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # set in Colab secrets\n\n# Then run the script:\n# exec(open("hate_speech_detection_llama.py").read())\n\nNOTE: You need a HuggingFace account and access to LLaMA 3.1:\n  1. Go to https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct\n  2. Click "Request access" and wait for approval (usually < 1 hour)\n  3. Generate a token at https://huggingface.co/settings/tokens\n  4. Store it as HF_TOKEN in Colab Secrets\n\nALTERNATIVE (no access needed):\n  Change MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"\n  Mistral is open-access and performs nearly identically.\n'